In [ ]:
# @title CELL 1: Install Dependencies
# Instal ModelScope sebagai mirror alternatif untuk mengatasi TimeoutError
!pip install -q modelscope

# Gunakan torch yang sudah ada di Kaggle
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# unsloth bisa pakai ini juga !pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q datasets sentencepiece protobuf

Installing Unsloth and dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 62.3 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 28.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.6/288.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.6/179.6 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 43.4 MB/s eta 0:00:00:00:010

In [ ]:
# @title CELL 2: Import Libraries
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow warnings
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # Use only first GPU

# Mengatasi TimeoutError dari HuggingFace
# Menginstruksikan Unsloth untuk menggunakan ModelScope sebagai alternatif
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

# Disable statistics check to avoid HuggingFace timeout
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'

import json
import re
import torch
import random
import numpy as np
from datetime import datetime
from typing import List, Dict, Tuple
import gc

from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForLanguageModeling

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


E0000 00:00:1765335486.548487      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765335486.604405      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


/usr/local/lib/python3.11/dist-packages/unsloth/models/rl_replacements.py:946: UserWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  import trl.experimental.openenv.utils as openenv_utils
[unsloth_zoo.log|WARNING]Unsloth: Failed to import trl openenv: No module named 'trl.experimental.openenv'


All libraries imported successfully!
Unsloth sekarang menggunakan ModelScope sebagai fallback.
Statistics check disabled to avoid timeout issues.


In [ ]:
# @title CELL 3: Check Environment
from importlib.metadata import version
import sys

print("Environment Check:")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")

Environment Check:
Python: 3.11.13
PyTorch: 2.6.0+cu124
CUDA Available: True
CUDA Version: 12.4
GPU Count: 1
  GPU 0: Tesla T4
  Memory: 14.7 GB


In [ ]:
# @title CELL 4: Dual Domain Configuration

class DualDomainConfig:
    """Configuration for both Restaurant and Laptop domains"""

    # Base settings
    BASE_DIR = "/kaggle/working"
    OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

    # Task settings
    TASK = "task2"
    SUBTASK = "subtask_2"
    LANGUAGE = "eng"

    # Model settings
    MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

        # PROMPT TEMPLATE SELECTION
    SELECTED_TEMPLATE = "one_shot_instruction"  # ganti prompt

    # Template-specific max_seq_length (auto-adjusted berdasarkan template)
    TEMPLATE_MAX_LENGTHS = {
        "zero_shot_instruction": 200,
        "one_shot_instruction": 232,
        "two_shot_instruction": 272,
        "code_zero_shot": 264,
        "code_one_shot": 328,
        "code_two_shot": 384,
    }

    # MAX_SEQ_LENGTH otomatis disesuaikan dengan template
    @property
    def MAX_SEQ_LENGTH(self):
        """Auto-adjust max_seq_length based on selected template"""
        return self.TEMPLATE_MAX_LENGTHS.get(
            self.SELECTED_TEMPLATE,
            384  # Fallback ke max jika template tidak dikenali
        )

    # LoRA settings
    LORA_R = 16
    LORA_ALPHA = 32
    LORA_DROPOUT = 0.0

    # Training settings
    BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = 16
    EPOCHS = 3
    LEARNING_RATE = 1e-4
    WARMUP_STEPS = 20

    SEED = 42

    @staticmethod
    def get_urls(domain: str):
        """Get URLs for specific domain"""
        return {
            'train': f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/{DualDomainConfig.SUBTASK}/{DualDomainConfig.LANGUAGE}/{DualDomainConfig.LANGUAGE}_{domain}_train_alltasks.jsonl",
            'test': f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/{DualDomainConfig.SUBTASK}/{DualDomainConfig.LANGUAGE}/{DualDomainConfig.LANGUAGE}_{domain}_dev_{DualDomainConfig.TASK}.jsonl"
        }

    @staticmethod
    def get_output_dir(domain: str):
        """Get output directory for specific domain"""
        return os.path.join(DualDomainConfig.OUTPUT_DIR, domain)

# Initialize config
config = DualDomainConfig()

# Set seeds
random.seed(config.SEED)
np.random.seed(config.SEED)
torch.manual_seed(config.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.SEED)

print(f"Domains supported: Restaurant, Laptop")
print(f"Model: {config.MODEL_NAME}")
print(f"\n  PROMPT SETTINGS:")
print(f"  Selected template: {config.SELECTED_TEMPLATE}")
print(f"  Auto max_seq_length: {config.MAX_SEQ_LENGTH} tokens")
print(f"\n  TRAINING SETTINGS:")
print(f"  Batch size: {config.BATCH_SIZE}")
print(f"  Gradient accumulation: {config.GRADIENT_ACCUMULATION_STEPS}")
print(f"  Effective batch: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS}")
print(f"  Epochs: {config.EPOCHS}")
print(f"  Learning rate: {config.LEARNING_RATE}")

Dual Domain Configuration:
Domains supported: Restaurant, Laptop
Model: unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit

  PROMPT SETTINGS:
  Selected template: one_shot_instruction
  Auto max_seq_length: 232 tokens

  TRAINING SETTINGS:
  Batch size: 1
  Gradient accumulation: 16
  Effective batch: 16
  Epochs: 3
  Learning rate: 0.0001


In [ ]:
# @title CELL 5: Create Directories
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
print(f"Created output directory: {config.OUTPUT_DIR}")

Created output directory: /kaggle/working/outputs


In [ ]:
# @title CELL 6: Download Datasets for BOTH Domains
import urllib.request
from urllib.error import URLError, HTTPError

def download_with_retry(url: str, filepath: str, max_retries: int = 3) -> bool:
    """Download file with retry mechanism"""
    for attempt in range(max_retries):
        try:
            print(f"Attempt {attempt + 1}/{max_retries}: Downloading from {url}")
            urllib.request.urlretrieve(url, filepath)
            print(f" Saved to: {filepath}")
            return True
        except (URLError, HTTPError) as e:
            print(f"Error: {str(e)}")
            if attempt < max_retries - 1:
                print(f"  Retrying...")
            else:
                print(f"  Failed after {max_retries} attempts")
                return False
        except Exception as e:
            print(f"Unexpected error: {str(e)}")
            return False
    return False

def download_domain_data(domain: str):
    """Download train and test data for a specific domain"""
    print(f"\nDOWNLOADING {domain.upper()} DATA")

    urls = config.get_urls(domain)

    train_file = os.path.join(config.BASE_DIR, f"eng_{domain}_train.jsonl")
    test_file = os.path.join(config.BASE_DIR, f"eng_{domain}_test.jsonl")

    success_train = download_with_retry(urls['train'], train_file)
    success_test = download_with_retry(urls['test'], test_file)

    if not (success_train and success_test):
        raise Exception(f"Failed to download {domain} datasets!")
    return train_file, test_file

# Create domain directories
for domain in ['restaurant', 'laptop']:
    domain_dir = config.get_output_dir(domain)
    os.makedirs(domain_dir, exist_ok=True)
    print(f" Created directory: {domain_dir}")

# Download Restaurant data
restaurant_train_file, restaurant_test_file = download_domain_data('restaurant')

# Download Laptop data
laptop_train_file, laptop_test_file = download_domain_data('laptop')

 Created directory: /kaggle/working/outputs/restaurant
 Created directory: /kaggle/working/outputs/laptop

STARTING DOWNLOADS

DOWNLOADING RESTAURANT DATA
Attempt 1/3: Downloading from https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/subtask_2/eng/eng_restaurant_train_alltasks.jsonl
 Saved to: /kaggle/working/eng_restaurant_train.jsonl
Attempt 1/3: Downloading from https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/subtask_2/eng/eng_restaurant_dev_task2.jsonl
 Saved to: /kaggle/working/eng_restaurant_test.jsonl
 Downloaded restaurant datasets successfully

DOWNLOADING LAPTOP DATA
Attempt 1/3: Downloading from https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/subtask_2/eng/eng_laptop_train_alltasks.jsonl
 Saved to: /kaggle/working/eng_laptop_train.jsonl
Attempt 1/3: Downloading from https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/subtask_2/eng/eng_laptop_dev_task2.js

In [ ]:
# @title CELL 7: Load Data for BOTH Domains
def load_jsonl(filepath: str) -> List[Dict]:
    """Load JSONL file"""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

# Load Restaurant data
print("\nLoading Restaurant datad")
restaurant_train_data = load_jsonl(restaurant_train_file)
restaurant_test_data = load_jsonl(restaurant_test_file)
print(f"   Train: {len(restaurant_train_data)} samples")
print(f"   Test:  {len(restaurant_test_data)} samples")

# Load Laptop data
print("\nLoading Laptop datad")
laptop_train_data = load_jsonl(laptop_train_file)
laptop_test_data = load_jsonl(laptop_test_file)
print(f"   Train: {len(laptop_train_data)} samples")
print(f"   Test:  {len(laptop_test_data)} samples")

# Show sample
print("\nSample Restaurant data:")
print(json.dumps(restaurant_train_data[0], indent=2))

print("\nSample Laptop data:")
print(json.dumps(laptop_train_data[0], indent=2))

LOADING DATASETS

Loading Restaurant data...
   Train: 2284 samples
   Test:  200 samples

Loading Laptop data...
   Train: 4076 samples
   Test:  200 samples

────────────────────────────────────────────────────────────────────────────────
Sample Restaurant data:
{
  "ID": "rest16_quad_dev_1",
  "Text": "ca n ' t wait wait for my next visit .",
  "Quadruplet": [
    {
      "Aspect": "NULL",
      "Opinion": "NULL",
      "Category": "RESTAURANT#GENERAL",
      "VA": "6.75#6.38"
    }
  ]
}

────────────────────────────────────────────────────────────────────────────────
Sample Laptop data:
{
  "ID": "laptop_quad_dev_1",
  "Text": "this unit is ` ` pretty ` ` and stylish , so my high school daughter was attracted to it for that reason .",
  "Quadruplet": [
    {
      "Aspect": "unit",
      "Category": "LAPTOP#DESIGN_FEATURES",
      "Opinion": "pretty",
      "VA": "7.12#7.12"
    },
    {
      "Aspect": "unit",
      "Category": "LAPTOP#DESIGN_FEATURES",
      "Opinion": "stylish"

In [ ]:
# @title CELL 7a: Data Cleaning and Validation
import pandas as pd

def check_and_clean_data(data: List[Dict], domain: str) -> Tuple[List[Dict], Dict]:
    """
    Check for null/empty values and duplicates, then clean the data
    Returns: cleaned data and statistics
    """
    print(f"\nDATA CLEANING - {domain.upper()}")

    initial_count = len(data)
    print(f"\n Initial data count: {initial_count}")

    # Convert to DataFrame for easier analysis
    df_data = []
    for idx, item in enumerate(data):
        df_data.append({
            'index': idx,
            'ID': item.get('ID', ''),
            'Text': item.get('Text', ''),
            'Quadruplet': item.get('Quadruplet', [])
        })
    df = pd.DataFrame(df_data)

    # 1. CHECK FOR NULL/EMPTY TEXT VALUES
    print("\n1. CHECKING NULL/EMPTY TEXT VALUES")

    # Check for various types of invalid Text
    null_text = df[
        df['Text'].isna() |                                    # pandas NA/None
        (df['Text'] == '') |                                   # empty string
        (df['Text'].str.strip() == '') |                       # whitespace only
        (df['Text'].str.upper() == 'NULL') |                   # "NULL" or "null"
        (df['Text'].str.upper() == 'NONE') |                   # "NONE" or "none"
        (df['Text'].str.lower() == 'nan')                      # "nan" or "NaN"
    ]

    print(f"\n Found issues:")
    print(f"   • Invalid Text entries: {len(null_text)}")

    # Show samples of problematic data
    if len(null_text) > 0:
        print(f"\n    Sample invalid Text entries:")
        for i, row in null_text.head(5).iterrows():
            print(f"      - ID: {row['ID']}, Text: '{row['Text']}'")

    # 2. CHECK FOR EMPTY QUADRUPLET
    print("\n2. CHECKING EMPTY QUADRUPLET")

    # Check empty Quadruplet
    empty_quad = df[df['Quadruplet'].apply(
        lambda x: len(x) == 0 if isinstance(x, list) else True
    )]

    print(f"\n Found issues:")
    print(f"   • Empty Quadruplet list: {len(empty_quad)}")

    if len(empty_quad) > 0:
        print(f"\n    Sample empty Quadruplet:")
        for i, row in empty_quad.head(3).iterrows():
            text_preview = row['Text'][:50] if len(row['Text']) > 50 else row['Text']
            print(f"      - ID: {row['ID']}, Text: '{text_preview}...'")

    # 3. REMOVE INVALID ENTRIES
    # Combine indices to remove
    indices_to_remove = set()
    indices_to_remove.update(null_text.index.tolist())
    indices_to_remove.update(empty_quad.index.tolist())

    # Remove problematic data
    cleaned_data = [data[i] for i in range(len(data)) if i not in indices_to_remove]
    after_null_clean = len(cleaned_data)
    removed_null = initial_count - after_null_clean

    print(f"\n Removed {removed_null} samples with invalid Text/Quadruplet")
    print(f"   Remaining: {after_null_clean}")

    # 4. CHECK FOR DUPLICATES
    print("\n3. CHECKING DUPLICATES (based on Text)")

    # Build text to indices mapping
    text_to_indices = {}
    for idx, item in enumerate(cleaned_data):
        text = item['Text'].strip().lower()
        if text not in text_to_indices:
            text_to_indices[text] = []
        text_to_indices[text].append(idx)

    # Find duplicates
    duplicates = {text: indices for text, indices in text_to_indices.items()
                  if len(indices) > 1}

    print(f"\n Found {len(duplicates)} unique texts with duplicates")
    print(f"   Total duplicate samples: {sum(len(v)-1 for v in duplicates.values())}")

    # Show duplicate samples
    if len(duplicates) > 0:
        print(f"\n    Sample duplicates (showing first 3):")
        for i, (text, indices) in enumerate(list(duplicates.items())[:3]):
            text_preview = text[:60] if len(text) > 60 else text
            print(f"\n      Duplicate {i+1}: '{text_preview}...'")
            print(f"      Appears {len(indices)} times at indices: {indices[:5]}")
            for idx in indices[:2]:  # Show first 2 occurrences
                print(f"         - ID: {cleaned_data[idx]['ID']}")

    # Remove duplicates (keep first occurrence)
    seen_texts = set()
    deduplicated_data = []
    for item in cleaned_data:
        text = item['Text'].strip().lower()
        if text not in seen_texts:
            seen_texts.add(text)
            deduplicated_data.append(item)

    after_dedup = len(deduplicated_data)
    removed_dup = after_null_clean - after_dedup

    print(f"\n Removed {removed_dup} duplicate samples")
    print(f"   Remaining: {after_dedup}")

    # 5. VERIFY ASPECT/OPINION "NULL" (Valid Cases)
    print("\n4. INFO: Aspect/Opinion with 'NULL' String")

    null_aspect_count = 0
    null_opinion_count = 0
    for item in deduplicated_data:
        for quad in item['Quadruplet']:
            if quad.get('Aspect', '').upper() == 'NULL':
                null_aspect_count += 1
            if quad.get('Opinion', '').upper() == 'NULL':
                null_opinion_count += 1

    print(f"\n Found (these are VALID, not errors):")
    print(f"   • Quadruplets with Aspect='NULL': {null_aspect_count}")
    print(f"   • Quadruplets with Opinion='NULL': {null_opinion_count}")
    print(f"      These represent general sentiment without specific aspect/opinion")

    # 6. SUMMARY
    print("\nCLEANING SUMMARY")
    print(f"Initial count:                {initial_count}")
    print(f"After removing invalid:       {after_null_clean} (-{removed_null})")
    print(f"After removing duplicates:    {after_dedup} (-{removed_dup})")
    print(f"Total removed:                {initial_count - after_dedup}")
    print(f"Retention rate:               {after_dedup/initial_count*100:.2f}%")

    stats = {
        'initial_count': initial_count,
        'after_null_clean': after_null_clean,
        'after_dedup': after_dedup,
        'removed_null': removed_null,
        'removed_dup': removed_dup,
        'total_removed': initial_count - after_dedup,
        'retention_rate': after_dedup/initial_count,
        'null_aspect_count': null_aspect_count,
        'null_opinion_count': null_opinion_count
    }

    return deduplicated_data, stats

# RUN DATA CLEANING
# Clean restaurant data
clean_restaurant_data, restaurant_stats = check_and_clean_data(
    restaurant_train_data,
    "Restaurant"
)

# Save cleaned data
clean_restaurant_file = os.path.join(config.BASE_DIR, "eng_restaurant_train_clean.jsonl")
with open(clean_restaurant_file, 'w', encoding='utf-8') as f:
    for item in clean_restaurant_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
print(f" Cleaned restaurant data saved to: {clean_restaurant_file}")

# Clean laptop data
clean_laptop_data, laptop_stats = check_and_clean_data(
    laptop_train_data,
    "Laptop"
)

# Save cleaned data
clean_laptop_file = os.path.join(config.BASE_DIR, "eng_laptop_train_clean.jsonl")
with open(clean_laptop_file, 'w', encoding='utf-8') as f:
    for item in clean_laptop_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
print(f" Cleaned laptop data saved to: {clean_laptop_file}")

# Overall summary
print("\nOVERALL CLEANING SUMMARY")
print(f"{'Domain':<15} {'Initial':<10} {'After Clean':<12} {'Removed':<10} {'Retention':<12}")
print(f"{'Restaurant':<15} {restaurant_stats['initial_count']:<10} {restaurant_stats['after_dedup']:<12} {restaurant_stats['total_removed']:<10} {restaurant_stats['retention_rate']*100:<12.2f}%")
print(f"{'Laptop':<15} {laptop_stats['initial_count']:<10} {laptop_stats['after_dedup']:<12} {laptop_stats['total_removed']:<10} {laptop_stats['retention_rate']*100:<12.2f}%")



STARTING DATA CLEANING PROCESS



DATA CLEANING - RESTAURANT

 Initial data count: 2284

────────────────────────────────────────────────────────────────────────────────
1️⃣ CHECKING NULL/EMPTY TEXT VALUES
────────────────────────────────────────────────────────────────────────────────

 Found issues:
   • Invalid Text entries: 0

────────────────────────────────────────────────────────────────────────────────
2️⃣ CHECKING EMPTY QUADRUPLET
────────────────────────────────────────────────────────────────────────────────

 Found issues:
   • Empty Quadruplet list: 0

 Removed 0 samples with invalid Text/Quadruplet
   Remaining: 2284

────────────────────────────────────────────────────────────────────────────────
3️⃣ CHECKING DUPLICATES (based on Text)
────────────────────────────────────────────────────────────────────────────────

 Found 8 unique texts with duplicates
   Total duplicate samples: 8

    Sample duplicates (showing first 3):

      Duplicate 1: 'yum !...'
      Appears 

In [ ]:
# @title CELL 8: Enhanced Prompt Template Library (6 Variants)
from typing import List, Dict

# INSTRUCTION-BASED TEMPLATES
TEMPLATE_ZERO_SHOT_INSTRUCTION = """Extract (Aspect, Opinion, V#A) triplets from text.
- Aspect: entity/feature mentioned
- Opinion: sentiment expression
- V#A: Valence (1-9: neg→pos) # Arousal (1-9: calm→excited)

Text: {text}

Output triplets:"""


TEMPLATE_ONE_SHOT_INSTRUCTION = """Extract (Aspect, Opinion, V#A) triplets.
V#A format: Valence (1-9: neg→pos) # Arousal (1-9: calm→excited)

Example:
Text: average to good thai food, but terrible delivery.
Output: (thai food, average to good, 6.75#6.38), (delivery, terrible, 2.88#6.62)

Text: {text}
Output:"""


TEMPLATE_TWO_SHOT_INSTRUCTION = """Extract (Aspect, Opinion, V#A) triplets.
V#A: Valence (1-9) # Arousal (1-9)

Example 1:
Text: average to good thai food, but terrible delivery.
Output: (thai food, average to good, 6.75#6.38), (delivery, terrible, 2.88#6.62)

Example 2:
Text: Great performance at a great price
Output: (performance, great, 7.88#7.50), (price, great, 7.88#7.50)

Text: {text}
Output:"""

# CODE-STYLE TEMPLATES (Following Li et al. 2023)
TEMPLATE_CODE_ZERO_SHOT = """# According to the sentence, extract aspect-opinion-sentiment triplets
def TRIPLET_EXTRACTION(sentence):
    # aspect is the entity or feature mentioned in the sentence
    # opinion is the sentiment expression describing the aspect
    # sentiment is in V#A format where V=valence(1-9), A=arousal(1-9)
    triplet_list = <|fim_hole|>

    for aspect, opinion, sentiment in triplet_list:
        assert aspect in sentence
        assert opinion in sentence
        assert "#" in sentiment

    return triplet_list

print(TRIPLET_EXTRACTION("{text}"))"""


TEMPLATE_CODE_ONE_SHOT = """# According to the sentence, extract aspect-opinion-sentiment triplets
def TRIPLET_EXTRACTION(sentence):
    # aspect is the entity or feature mentioned in the sentence
    # opinion is the sentiment expression describing the aspect
    # sentiment is in V#A format where V=valence(1-9), A=arousal(1-9)
    triplet_list = <|fim_hole|>

    for aspect, opinion, sentiment in triplet_list:
        assert aspect in sentence
        assert opinion in sentence
        assert "#" in sentiment

    return triplet_list

# Example
assert TRIPLET_EXTRACTION("average to good thai food, but terrible delivery.") == [
    ("thai food", "average to good", "6.75#6.38"),
    ("delivery", "terrible", "2.88#6.62")
]

print(TRIPLET_EXTRACTION("{text}"))"""


TEMPLATE_CODE_TWO_SHOT = """# According to the sentence, extract aspect-opinion-sentiment triplets
def TRIPLET_EXTRACTION(sentence):
    # aspect is the entity or feature mentioned in the sentence
    # opinion is the sentiment expression describing the aspect
    # sentiment is in V#A format where V=valence(1-9), A=arousal(1-9)
    triplet_list = <|fim_hole|>

    for aspect, opinion, sentiment in triplet_list:
        assert aspect in sentence
        assert opinion in sentence
        assert "#" in sentiment

    return triplet_list

# Example 1
assert TRIPLET_EXTRACTION("average to good thai food, but terrible delivery.") == [
    ("thai food", "average to good", "6.75#6.38"),
    ("delivery", "terrible", "2.88#6.62")
]

# Example 2
assert TRIPLET_EXTRACTION("Great performance at a great price") == [
    ("performance", "great", "7.88#7.50"),
    ("price", "great", "7.88#7.50")
]

print(TRIPLET_EXTRACTION("{text}"))"""


# HELPER FUNCTIONS
def create_prompt(text: str, template_type: str = "one_shot_instruction") -> str:
    """
    Create extraction prompt using selected template.

    Args:
        text: Input text to extract triplets from
        template_type: One of:
            - "zero_shot_instruction" - Basic instruction without examples
            - "one_shot_instruction" - One example (default)
            - "two_shot_instruction" - Two examples
            - "code_zero_shot" - Code-style without examples
            - "code_one_shot" - Code-style with one example
            - "code_two_shot" - Code-style with two examples

    Returns:
        Formatted prompt string
    """
    templates = {
        "zero_shot_instruction": TEMPLATE_ZERO_SHOT_INSTRUCTION,
        "one_shot_instruction": TEMPLATE_ONE_SHOT_INSTRUCTION,
        "two_shot_instruction": TEMPLATE_TWO_SHOT_INSTRUCTION,
        "code_zero_shot": TEMPLATE_CODE_ZERO_SHOT,
        "code_one_shot": TEMPLATE_CODE_ONE_SHOT,
        "code_two_shot": TEMPLATE_CODE_TWO_SHOT,
    }

    if template_type not in templates:
        raise ValueError(f"Unknown template type: {template_type}. Choose from: {list(templates.keys())}")

    return templates[template_type].format(text=text)


def format_triplet_output(quadruplets: List[Dict]) -> str:
    """
    Format quadruplets to triplet output (ignore Category).

    Args:
        quadruplets: List of dicts with 'Aspect', 'Opinion', 'VA', 'Category'

    Returns:
        Formatted string: "(aspect1, opinion1, va1), (aspect2, opinion2, va2), ..."
    """
    triplets = []
    for quad in quadruplets:
        aspect = quad['Aspect']
        opinion = quad['Opinion']
        va = quad['VA']
        triplets.append(f"({aspect}, {opinion}, {va})")
    return ", ".join(triplets) if triplets else "No triplets"


def get_max_length_recommendation(template_type: str) -> int:
    """
    Get recommended max_length for each template type.

    Args:
        template_type: Template name

    Returns:
        Recommended max_length in tokens
    """
    # Based on prompt size analysis
    recommendations = {
    "zero_shot_instruction": 200,
    "one_shot_instruction": 232,
    "two_shot_instruction": 272,
    "code_zero_shot": 264,
    "code_one_shot": 328,
    "code_two_shot": 384,
    }
    return recommendations.get(template_type, 512)


def analyze_all_templates(sample_text: str = "The food was delicious but service was slow"):
    """
    Analyze all templates and show statistics.

    Args:
        sample_text: Text to use for analysis
    """
    print("\nTEMPLATE ANALYSIS - ALL VARIANTS")


    print(f"\n  Prompt Statistics (with sample text: '{sample_text}'):\n")
    print(f"{'Template':<25} {'Chars':<8} {'Words':<8} {'Tokens*':<10} {'Max Length':<12}")


    templates = [
        "zero_shot_instruction",
        "one_shot_instruction",
        "two_shot_instruction",
        "code_zero_shot",
        "code_one_shot",
        "code_two_shot"
    ]

    for template in templates:
        prompt = create_prompt(sample_text, template)
        words = len(prompt.split())
        tokens_est = int(words * 1.3)  # Rough estimate
        max_len = get_max_length_recommendation(template)

        print(f"{template:<25} {len(prompt):<8} {words:<8} {tokens_est:<10} {max_len:<12}")

    print("\n* Token estimation: words × 1.3 (rough approximation)")

# CONFIGURATION & DEMO
if __name__ == "__main__":
    # Set your preferred template here:
    SELECTED_TEMPLATE = "one_shot_instruction"  # Change this to test different prompts

    print(f"\nCurrently selected: {SELECTED_TEMPLATE}")


    # Show example of selected template
    sample_text = "The food was delicious but service was slow"
    sample_prompt = create_prompt(sample_text, SELECTED_TEMPLATE)

    print(f"\n  Sample prompt preview with '{SELECTED_TEMPLATE}':")
    print(sample_prompt)

    # Quick stats
    words = len(sample_prompt.split())
    tokens_est = int(words * 1.3)
    max_len = get_max_length_recommendation(SELECTED_TEMPLATE)

    print(f"\n Prompt Statistics:")
    print(f"  Total length: {len(sample_prompt)} characters")
    print(f"  Word count: {words} words")
    print(f"  Estimated tokens: ~{tokens_est}")
    print(f"  Recommended max_length: {max_len}")

    # Analyze all templates
    analyze_all_templates(sample_text)


✅ Currently selected: one_shot_instruction

  Sample prompt preview with 'one_shot_instruction':
────────────────────────────────────────────────────────────────────────────────
Extract (Aspect, Opinion, V#A) triplets.
V#A format: Valence (1-9: neg→pos) # Arousal (1-9: calm→excited)

Example:
Text: average to good thai food, but terrible delivery.
Output: (thai food, average to good, 6.75#6.38), (delivery, terrible, 2.88#6.62)

Text: The food was delicious but service was slow
Output:
────────────────────────────────────────────────────────────────────────────────

 Prompt Statistics:
  Total length: 311 characters
  Word count: 44 words
  Estimated tokens: ~57
  Recommended max_length: 232

TEMPLATE ANALYSIS - ALL VARIANTS

  Prompt Statistics (with sample text: 'The food was delicious but service was slow'):

Template                  Chars    Words    Tokens*    Max Length  
--------------------------------------------------------------------------------
zero_shot_instruction     2

In [ ]:
# @title CELL 9: Load Tokenizer Only
gc.collect()
torch.cuda.empty_cache()

# Only load tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

# Proper tokenizer setup
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"

print(f"Tokenizer loaded: {config.MODEL_NAME}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
print(f"Padding side: {tokenizer.padding_side}")

Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

Tokenizer loaded: unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit
Vocab size: 151665
Pad token: <|im_end|> (ID: 151645)
EOS token: <|im_end|> (ID: 151645)
Padding side: right

Note: Model will be loaded separately during training for each domain


In [ ]:
# @title CELL 9a: Analyze Token Statistics to Determine Optimal max_length
import numpy as np
from typing import List, Dict

print("TOKEN STATISTICS ANALYSIS")
print("\nThis will analyze your actual data to determine optimal max_length")
print("for each prompt template.\n")

def analyze_tokens_for_template(data: List[Dict], tokenizer, template_type: str,
                                 domain: str) -> Dict:
    """Analyze token lengths for a specific template"""

    total_lengths = []

    for sample in data:
        text = sample['Text']
        quadruplets = sample.get('Quadruplet', [])

        # Create prompt and answer
        prompt = create_prompt(text, template_type)
        answer = format_triplet_output(quadruplets)

        # Full text in Qwen format
        full_text = f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n{answer}<|im_end|>"

        # Tokenize
        tokens = tokenizer(full_text, truncation=False, add_special_tokens=False)
        total_lengths.append(len(tokens['input_ids']))

    # Calculate statistics
    total_lengths = np.array(total_lengths)

    stats = {
        'template': template_type,
        'domain': domain,
        'count': len(data),
        'min': int(total_lengths.min()),
        'max': int(total_lengths.max()),
        'mean': float(total_lengths.mean()),
        'median': float(np.median(total_lengths)),
        'p95': float(np.percentile(total_lengths, 95)),
        'p99': float(np.percentile(total_lengths, 99)),
        'lengths': total_lengths
    }

    return stats

# Templates to analyze
templates = [
    "zero_shot_instruction",
    "one_shot_instruction",
    "two_shot_instruction",
    "code_zero_shot",
    "code_one_shot",
    "code_two_shot"
]

# Store results
all_stats = {}
# Analyze each template for both domains
for template in templates:
    print(f"Analyzing: {template}")

    # Restaurant
    rest_stats = analyze_tokens_for_template(
        clean_restaurant_data, tokenizer, template, "restaurant"
    )

    # Laptop
    lap_stats = analyze_tokens_for_template(
        clean_laptop_data, tokenizer, template, "laptop"
    )

    # Take the max of both domains for universal coverage
    max_p99 = max(rest_stats['p99'], lap_stats['p99'])
    max_max = max(rest_stats['max'], lap_stats['max'])

    all_stats[template] = {
        'restaurant': rest_stats,
        'laptop': lap_stats,
        'universal_p99': max_p99,
        'universal_max': max_max
    }

    print(f"  Restaurant: mean={rest_stats['mean']:.1f}, p95={rest_stats['p95']:.1f}, p99={rest_stats['p99']:.1f}, max={rest_stats['max']}")
    print(f"  Laptop:     mean={lap_stats['mean']:.1f}, p95={lap_stats['p95']:.1f}, p99={lap_stats['p99']:.1f}, max={lap_stats['max']}")
    print(f"  Universal:  p99={max_p99:.1f}, max={max_max}\n")

# GENERATE RECOMMENDATIONS BASED ON ACTUAL DATA
print("GENERATING RECOMMENDATIONS BASED ON YOUR DATA")
print(f"{'Template':<25} {'Mean':<10} {'P95':<10} {'P99':<10} {'Max':<10} {'Recommended':<15}")

recommended_max_lengths = {}

for template in templates:
    stats = all_stats[template]

    # Use p99 as base, round up to nearest multiple of 8 (GPU-friendly)
    recommended = int(stats['universal_p99'])
    recommended = ((recommended + 7) // 8) * 8

    # Calculate average stats
    avg_mean = (stats['restaurant']['mean'] + stats['laptop']['mean']) / 2
    avg_p95 = (stats['restaurant']['p95'] + stats['laptop']['p95']) / 2

    print(f"{template:<25} {avg_mean:<10.1f} {avg_p95:<10.1f} "
          f"{stats['universal_p99']:<10.1f} {stats['universal_max']:<10} "
          f"{recommended:<15}")

    recommended_max_lengths[template] = recommended

# COVERAGE CHECK
print(" DATA COVERAGE CHECK")

for template in templates:
    stats = all_stats[template]
    recommended = recommended_max_lengths[template]

    # Check coverage for both domains
    rest_coverage = np.sum(stats['restaurant']['lengths'] <= recommended) / len(stats['restaurant']['lengths']) * 100
    lap_coverage = np.sum(stats['laptop']['lengths'] <= recommended) / len(stats['laptop']['lengths']) * 100

    print(f"{template:<25} max_length={recommended:<5} → "
          f"Restaurant: {rest_coverage:>5.1f}%, Laptop: {lap_coverage:>5.1f}%")

# FINAL RECOMMENDATIONS
print("FINAL RECOMMENDATIONS")
print("# Update the get_max_length_recommendation() function with these values:")
print("# (Based on 99th percentile, rounded to multiple of 8)\n")

print("recommended_max_lengths = {")
for template, max_len in recommended_max_lengths.items():
    print(f'    "{template}": {max_len},')
print("}")


# UNIVERSAL MAX_LENGTH (for all templates)
universal_max = max(recommended_max_lengths.values())

print(f"   \nIf you want ONE max_length for ALL templates:")
print(f"   MAX_SEQ_LENGTH = {universal_max}")
print(f"   (This covers 99% of data across all templates and both domains)")

# SAVE RESULTS TO FILE
import json

analysis_results = {
    'templates_analyzed': templates,
    'domains': ['restaurant', 'laptop'],
    'restaurant_samples': len(clean_restaurant_data),
    'laptop_samples': len(clean_laptop_data),
    'recommended_max_lengths': recommended_max_lengths,
    'universal_max_length': universal_max,
    'detailed_stats': {
        template: {
            'restaurant': {
                'mean': float(all_stats[template]['restaurant']['mean']),
                'p95': float(all_stats[template]['restaurant']['p95']),
                'p99': float(all_stats[template]['restaurant']['p99']),
                'max': int(all_stats[template]['restaurant']['max'])
            },
            'laptop': {
                'mean': float(all_stats[template]['laptop']['mean']),
                'p95': float(all_stats[template]['laptop']['p95']),
                'p99': float(all_stats[template]['laptop']['p99']),
                'max': int(all_stats[template]['laptop']['max'])
            }
        }
        for template in templates
    }
}

results_file = "/kaggle/working/token_analysis_results.json"
with open(results_file, 'w') as f:
    json.dump(analysis_results, f, indent=2)

print(f" Results saved to: {results_file}")


TOKEN STATISTICS ANALYSIS

This will analyze your actual data to determine optimal max_length
for each prompt template.

ANALYZING ALL TEMPLATES

Analyzing: zero_shot_instruction...
  Restaurant: mean=117.0, p95=159.0, p99=193.2, max=616
  Laptop:     mean=114.0, p95=150.0, p99=188.5, max=294
  Universal:  p99=193.2, max=616

Analyzing: one_shot_instruction...
  Restaurant: mean=151.0, p95=193.0, p99=227.2, max=650
  Laptop:     mean=148.0, p95=184.0, p99=222.5, max=328
  Universal:  p99=227.2, max=650

Analyzing: two_shot_instruction...
  Restaurant: mean=190.0, p95=232.0, p99=266.2, max=689
  Laptop:     mean=187.0, p95=223.0, p99=261.5, max=367
  Universal:  p99=266.2, max=689

Analyzing: code_zero_shot...
  Restaurant: mean=182.0, p95=225.0, p99=258.2, max=681
  Laptop:     mean=179.0, p95=215.0, p99=253.5, max=359
  Universal:  p99=258.2, max=681

Analyzing: code_one_shot...
  Restaurant: mean=246.0, p95=289.0, p99=322.2, max=745
  Laptop:     mean=243.0, p95=279.0, p99=317.5, ma

In [ ]:
# @title CELL 10: Prepare Training Data

def convert_to_training_format(data: List[Dict], config, tokenizer) -> Tuple[List[Dict], int]:
    """
    Convert raw data to training format with length filtering.
    Otomatis menggunakan template dari config.SELECTED_TEMPLATE
    """
    formatted_data = []
    skipped = 0

    # Ambil template dari config
    template_type = config.SELECTED_TEMPLATE
    max_length = config.MAX_SEQ_LENGTH

    print(f"  Using template: {template_type}")
    print(f"  Max sequence length: {max_length}")

    for sample in data:
        text = sample['Text']
        quadruplets = sample.get('Quadruplet', [])

        # Gunakan template yang dipilih di config
        prompt = create_prompt(text, template_type)
        answer = format_triplet_output(quadruplets)

        # Format untuk Qwen chat template
        full_text = f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n{answer}<|im_end|>"

        # Check length BEFORE adding
        test_tokens = tokenizer(full_text, truncation=False, add_special_tokens=False)
        if len(test_tokens['input_ids']) > max_length:
            skipped += 1
            continue

        formatted_sample = {"text": full_text}
        formatted_data.append(formatted_sample)

    return formatted_data, skipped

print(f"   Will use template: {config.SELECTED_TEMPLATE}")
print(f"   Max sequence length: {config.MAX_SEQ_LENGTH}")

✅ Training data preparation function ready
   Will use template: one_shot_instruction
   Max sequence length: 232


In [ ]:
# @title CELL 11: TRAINING FUNCTION

def train_domain_model(domain: str, train_data: List[Dict],
                       config, tokenizer, base_model_name: str):
    """
    Train model for specific domain.
    Otomatis menggunakan MAX_SEQ_LENGTH dari config.
    Returns: trained model, training stats
    """
    import time
    from datetime import datetime

    print(f"TRAINING MODEL - {domain.upper()}")

    # Prepare training data (updated signature)
    print(f"\nPreparing {domain} training data")
    formatted_train_data, skipped = convert_to_training_format(
        train_data, config, tokenizer
    )
    train_dataset = Dataset.from_list(formatted_train_data)

    print(f"  Converted {len(train_dataset)} samples")
    print(f"  Skipped: {skipped} samples (too long)")
    print(f"  Retention: {len(train_dataset)/len(train_data)*100:.1f}%")

    # Load model
    print(f"\nLoading model: {base_model_name}")
    gc.collect()
    torch.cuda.empty_cache()

    model, _ = FastLanguageModel.from_pretrained(
        model_name=base_model_name,
        max_seq_length=config.MAX_SEQ_LENGTH,  # ← Auto dari config
        dtype=None,
        load_in_4bit=True,
    )

    # Apply LoRA
    print("Applying LoRA")
    model = FastLanguageModel.get_peft_model(
        model,
        r=config.LORA_R,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=config.LORA_ALPHA,
        lora_dropout=config.LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=config.SEED,
    )

    # Training arguments
    domain_output_dir = config.get_output_dir(domain)

    training_args = TrainingArguments(
        per_device_train_batch_size=config.BATCH_SIZE,
        gradient_accumulation_steps=config.GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=config.EPOCHS,
        learning_rate=config.LEARNING_RATE,
        warmup_steps=config.WARMUP_STEPS,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        weight_decay=0.01,
        max_grad_norm=1.0,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        logging_first_step=True,
        output_dir=domain_output_dir,
        save_strategy="epoch",
        save_total_limit=1,
        gradient_checkpointing=True,
        seed=config.SEED,
        report_to="none",
        ddp_find_unused_parameters=False,
    )

    # Data collator
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
        pad_to_multiple_of=8
    )

    # Initialize trainer
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        dataset_text_field="text",
        max_seq_length=config.MAX_SEQ_LENGTH,  # ← Auto dari config
        data_collator=data_collator,
        packing=False,
        dataset_num_proc=1,
        args=training_args,
    )

    print("Training Configuration:")
    print(f"  Template: {config.SELECTED_TEMPLATE}")
    print(f"  Max seq length: {config.MAX_SEQ_LENGTH}")
    print(f"  Dataset size: {len(train_dataset)}")
    print(f"  Epochs: {config.EPOCHS}")
    print(f"  Batch size: {config.BATCH_SIZE}")
    print(f"  Gradient accumulation: {config.GRADIENT_ACCUMULATION_STEPS}")
    print(f"  Effective batch: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS}")
    print(f"{'─'*80}\n")

    # START TRAINING
    start_time = time.time()
    start_datetime = datetime.now()

    print(f"Started: {start_datetime.strftime('%Y-%m-%d %H:%M:%S')}\n")

    try:
        trainer_stats = trainer.train()

        # Calculate duration
        end_time = time.time()
        end_datetime = datetime.now()
        duration_seconds = end_time - start_time

        hours = int(duration_seconds // 3600)
        minutes = int((duration_seconds % 3600) // 60)
        seconds = int(duration_seconds % 60)


        print(f"  {domain.upper()} TRAINING COMPLETED!")

        print(f"Started:  {start_datetime.strftime('%H:%M:%S')}")
        print(f"Finished: {end_datetime.strftime('%H:%M:%S')}")
        print(f"Duration: {hours}h {minutes}m {seconds}s")
        print(f"Steps: {trainer_stats.global_step}")
        print(f"Final Loss: {trainer_stats.training_loss:.4f}")


        # Save model
        best_path = os.path.join(domain_output_dir, "best_model")
        trainer.save_model(best_path)
        tokenizer.save_pretrained(best_path)
        print(f"\n  Model saved to: {best_path}")

        return model, {
            'duration_seconds': duration_seconds,
            'duration_formatted': f"{hours}h {minutes}m {seconds}s",
            'steps': trainer_stats.global_step,
            'final_loss': trainer_stats.training_loss,
            'start_time': start_datetime,
            'end_time': end_datetime
        }

    except Exception as e:

        print(f"❌ {domain.upper()} TRAINING FAILED!")

        print(f"Error: {str(e)}\n")

        import traceback
        traceback.print_exc()
        raise

    finally:
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
# @title CELL 12: TRAIN RESTAURANT MODEL
print("RESTAURANT DOMAIN TRAINING")
restaurant_model, restaurant_stats = train_domain_model(
    domain='restaurant',
    train_data=clean_restaurant_data,  # ← Gunakan data yang sudah dibersihkan
    config=config,
    tokenizer=tokenizer,
    base_model_name=config.MODEL_NAME
)

print("\n  Restaurant Training Summary:")
print(f"   Duration: {restaurant_stats['duration_formatted']}")
print(f"   Final Loss: {restaurant_stats['final_loss']:.4f}")
print(f"   Total Steps: {restaurant_stats['steps']}")



RESTAURANT DOMAIN TRAINING



TRAINING MODEL - RESTAURANT

Preparing restaurant training data...
  Using template: one_shot_instruction
  Max sequence length: 232
  Converted 2258 samples
  Skipped: 18 samples (too long)
  Retention: 99.2%

Loading model: unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit


2025-12-10 02:59:15,440 - modelscope - INFO - Got 11 files, start to download ...


Processing 11 items:   0%|          | 0.00/11.0 [00:00<?, ?it/s]

2025-12-10 03:01:20,096 - modelscope - INFO - Download model 'unsloth/qwen2.5-1.5b-instruct-bnb-4bit' successfully.
2025-12-10 03:01:20,097 - modelscope - INFO - Creating symbolic link [/root/.cache/modelscope/hub/models/unsloth/qwen2.5-1.5b-instruct-bnb-4bit].


==((====))==  Unsloth 2025.12.1: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Applying LoRA...


...

The model is already on multiple devices. Skipping the move to device specified in `args`.



────────────────────────────────────────────────────────────────────────────────
Training Configuration:
  Template: one_shot_instruction
  Max seq length: 232
  Dataset size: 2258
  Epochs: 3
  Batch size: 1
  Gradient accumulation: 16
  Effective batch: 16
────────────────────────────────────────────────────────────────────────────────

Started: 2025-12-10 03:01:33



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,258 | Num Epochs = 3 | Total steps = 426
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
1,4.122500
10,3.289900
20,1.918700
30,0.715400
40,0.471400
50,0.459400
60,0.437300
70,0.452100
80,0.438400
90,0.398500



✅ RESTAURANT TRAINING COMPLETED!
Started:  03:01:33
Finished: 03:48:48
Duration: 0h 47m 14s
Steps: 426
Final Loss: 0.5109

  Model saved to: /kaggle/working/outputs/restaurant/best_model

  Restaurant Training Summary:
   Duration: 0h 47m 14s
   Final Loss: 0.5109
   Total Steps: 426


In [ ]:
# @title CELL 13: TRAIN LAPTOP MODEL
print("LAPTOP DOMAIN TRAINING")

laptop_model, laptop_stats = train_domain_model(
    domain='laptop',
    train_data=clean_laptop_data,  # ← Gunakan data yang sudah dibersihkan
    config=config,
    tokenizer=tokenizer,
    base_model_name=config.MODEL_NAME
)

print("\n Laptop Training Summary:")
print(f"   Duration: {laptop_stats['duration_formatted']}")
print(f"   Final Loss: {laptop_stats['final_loss']:.4f}")
print(f"   Total Steps: {laptop_stats['steps']}")

# COMPARISON SUMMARY
print(" TRAINING COMPARISON - RESTAURANT vs LAPTOP")
print(f"\n{'Metric':<30} {'Restaurant':<20} {'Laptop':<20}")
print(f"{'Duration':<30} {restaurant_stats['duration_formatted']:<20} {laptop_stats['duration_formatted']:<20}")
print(f"{'Final Loss':<30} {restaurant_stats['final_loss']:<20.4f} {laptop_stats['final_loss']:<20.4f}")
print(f"{'Total Steps':<30} {restaurant_stats['steps']:<20} {laptop_stats['steps']:<20}")
print(f"{'Training Samples':<30} {len(clean_restaurant_data):<20} {len(clean_laptop_data):<20}")



LAPTOP DOMAIN TRAINING



TRAINING MODEL - LAPTOP

Preparing laptop training data...
  Using template: one_shot_instruction
  Max sequence length: 232
  Converted 4030 samples
  Skipped: 22 samples (too long)
  Retention: 99.5%

Loading model: unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit


2025-12-10 03:48:54,946 - modelscope - INFO - Target directory already exists, skipping creation.


==((====))==  Unsloth 2025.12.1: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Applying LoRA...


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/4030 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.



────────────────────────────────────────────────────────────────────────────────
Training Configuration:
  Template: one_shot_instruction
  Max seq length: 232
  Dataset size: 4030
  Epochs: 3
  Batch size: 1
  Gradient accumulation: 16
  Effective batch: 16
────────────────────────────────────────────────────────────────────────────────

Started: 2025-12-10 03:49:10



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,030 | Num Epochs = 3 | Total steps = 756
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
1,4.113100
10,3.384000
20,2.005400
30,0.743100
40,0.507600
50,0.506800
60,0.455800
70,0.465300
80,0.439700
90,0.433800



✅ LAPTOP TRAINING COMPLETED!
Started:  03:49:10
Finished: 05:13:17
Duration: 1h 24m 6s
Steps: 756
Final Loss: 0.4616

  Model saved to: /kaggle/working/outputs/laptop/best_model

 Laptop Training Summary:
   Duration: 1h 24m 6s
   Final Loss: 0.4616
   Total Steps: 756

 TRAINING COMPARISON - RESTAURANT vs LAPTOP

Metric                         Restaurant           Laptop              
────────────────────────────────────────────────────────────────────────────────
Duration                       0h 47m 14s           1h 24m 6s           
Final Loss                     0.5109               0.4616              
Total Steps                    426                  756                 
Training Samples               2276                 4052                


In [ ]:
# @title CELL 14: Inference Function
def extract_triplets(text: str) -> List[Dict]:
    """Extract triplets from model output"""
    pattern = r'\(([^,()]+),\s*([^,()]+),\s*([\d.]+#[\d.]+)\)'
    matches = re.findall(pattern, text)

    triplets = []
    for aspect, opinion, va in matches:
        try:
            v, a = map(float, va.split('#'))
            v = max(1.0, min(9.0, v))
            a = max(1.0, min(9.0, a))
            triplets.append({
                "Aspect": aspect.strip(),
                "Opinion": opinion.strip(),
                "VA": f"{v:.2f}#{a:.2f}"
            })
        except:
            continue
    return triplets

Inference function ready


In [ ]:
# @title CELL 15: Run Inference on Test Data (dual domain)
def run_inference_on_test(domain: str, test_data: List[Dict], model,
                          tokenizer, config):
    """Run inference on test data for specific domain"""

    print(f"RUNNING INFERENCE - {domain.upper()} TEST DATA")


    # Load best model for this domain
    best_path = os.path.join(config.get_output_dir(domain), "best_model")
    if os.path.exists(best_path):
        print(f"Loading best {domain} model from: {best_path}")
        model = FastLanguageModel.from_pretrained(
            best_path,
            max_seq_length=config.MAX_SEQ_LENGTH,
            dtype=None,
            load_in_4bit=True,
        )[0]

    FastLanguageModel.for_inference(model)
    model.eval()

    results = []

    print(f"Processing {len(test_data)} test samples")

    from tqdm.auto import tqdm

    for sample in tqdm(test_data, desc=f"{domain.capitalize()} inference"):
        text = sample['Text']
        prompt = create_prompt(text)

        messages = [{"role": "user", "content": prompt}]
        formatted = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            formatted,
            return_tensors="pt",
            truncation=True,
            max_length=config.MAX_SEQ_LENGTH,
            padding=True
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_new_tokens=256,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        if "assistant" in response:
            answer = response.split("assistant")[-1].strip()
        else:
            answer = response[len(formatted):].strip()

        triplets = extract_triplets(answer)

        results.append({
            "ID": sample['ID'],
            "Triplet": triplets
        })

    print(f"Inference completed: {len(results)} predictions")

    # Save predictions
    domain_output_dir = config.get_output_dir(domain)
    pred_file = os.path.join(domain_output_dir, f"pred_eng_{domain}.jsonl")
    with open(pred_file, 'w', encoding='utf-8') as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

    print(f"  Saved to: {pred_file}")

    return results, pred_file

# Run Inference for RESTAURANT
print("\n  RESTAURANT TEST INFERENCE")
restaurant_results, restaurant_pred_file = run_inference_on_test(
    domain='restaurant',
    test_data=restaurant_test_data,
    model=restaurant_model,
    tokenizer=tokenizer,
    config=config
)

# Run Inference for LAPTOP
print("\n LAPTOP TEST INFERENCE")
laptop_results, laptop_pred_file = run_inference_on_test(
    domain='laptop',
    test_data=laptop_test_data,
    model=laptop_model,
    tokenizer=tokenizer,
    config=config
)

print(" ALL INFERENCES COMPLETED")
print(f"Restaurant predictions: {len(restaurant_results)}")
print(f"Laptop predictions: {len(laptop_results)}")


  RESTAURANT TEST INFERENCE

RUNNING INFERENCE - RESTAURANT TEST DATA
Loading best restaurant model from: /kaggle/working/outputs/restaurant/best_model
==((====))==  Unsloth 2025.12.1: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
/root/.cache/modelscope/hub/models/unsloth/qwen2___5-1___5b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|vision_pad|>.
Processing 200 test samples...


Restaurant inference:   0%|          | 0/200 [00:00<?, ?it/s]

Inference completed: 200 predictions
✓ Saved to: /kaggle/working/outputs/restaurant/pred_eng_restaurant.jsonl

 LAPTOP TEST INFERENCE

RUNNING INFERENCE - LAPTOP TEST DATA
Loading best laptop model from: /kaggle/working/outputs/laptop/best_model
==((====))==  Unsloth 2025.12.1: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
/root/.cache/modelscope/hub/models/unsloth/qwen2___5-1___5b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|vision_pad|>.
Processing 200 test samples...


Laptop inference:   0%|          | 0/200 [00:00<?, ?it/s]

Inference completed: 200 predictions
✓ Saved to: /kaggle/working/outputs/laptop/pred_eng_laptop.jsonl

 ALL INFERENCES COMPLETED
Restaurant predictions: 200
Laptop predictions: 200


In [ ]:
# @title CELL 16: Show Sample Predictions (dual domain)
def show_sample_predictions(results: List[Dict], domain: str, n_samples: int = 3):
    """Show sample predictions for a domain"""

    print(f"SAMPLE PREDICTIONS - {domain.upper()}")


    for i in range(min(n_samples, len(results))):
        print(f"\nSample {i+1}:")
        print(f"  ID: {results[i]['ID']}")
        print(f"  Predictions: {len(results[i]['Triplet'])} triplets")
        for j, trip in enumerate(results[i]['Triplet'][:3]):
            print(f"    {j+1}. {trip}")
        if len(results[i]['Triplet']) > 3:
            print(f"    ... and {len(results[i]['Triplet'])-3} more")

# Show Restaurant samples
show_sample_predictions(restaurant_results, "Restaurant", 3)

# Show Laptop samples
show_sample_predictions(laptop_results, "Laptop", 3)


SAMPLE PREDICTIONS - RESTAURANT

Sample 1:
  ID: rest26_aste_dev_1
  Predictions: 2 triplets
    1. {'Aspect': 'food', 'Opinion': 'great', 'VA': '7.50#7.62'}
    2. {'Aspect': 'coffee', 'Opinion': 'great', 'VA': '7.50#7.62'}

Sample 2:
  ID: rest26_aste_dev_2
  Predictions: 2 triplets
    1. {'Aspect': 'customer service', 'Opinion': 'fantastic', 'VA': '7.50#7.50'}
    2. {'Aspect': 'food', 'Opinion': 'awesome', 'VA': '8.00#8.00'}

Sample 3:
  ID: rest26_aste_dev_3
  Predictions: 5 triplets
    1. {'Aspect': 'shrimp', 'Opinion': 'cooked perfectly', 'VA': '7.88#8.00'}
    2. {'Aspect': 'shrimp', 'Opinion': 'NULL', 'VA': '5.00#5.00'}
    3. {'Aspect': 'corn tortilla', 'Opinion': 'NULL', 'VA': '4.50#5.50'}
    ... and 2 more

SAMPLE PREDICTIONS - LAPTOP

Sample 1:
  ID: lap26_aste_dev_1
  Predictions: 2 triplets
    1. {'Aspect': 'perforemnce', 'Opinion': 'great', 'VA': '7.00#7.33'}
    2. {'Aspect': 'NULL', 'Opinion': 'great', 'VA': '7.50#7.83'}

Sample 2:
  ID: lap26_aste_dev_2
  Predic

In [ ]:
# @title CELL 17: Full Evaluation with AUTO-SAVE & AUTO-DOWNLOAD
import math
import zipfile
import shutil
from IPython.display import FileLink, display
import time

def evaluate_task2(gold_file: str, pred_file: str):
    """Official evaluation for Task 2"""

    print("Running evaluation metrics")
    # Load data
    with open(gold_file, 'r', encoding='utf-8') as f:
        gold_data = {}
        for line in f:
            if line.strip():
                entry = json.loads(line)
                gold_data[entry['ID']] = entry

    with open(pred_file, 'r', encoding='utf-8') as f:
        pred_data = {}
        for line in f:
            if line.strip():
                entry = json.loads(line)
                pred_data[entry['ID']] = entry

    print(f"Gold samples: {len(gold_data)}")
    print(f"Pred samples: {len(pred_data)}")

    # Metrics
    cTP_total = 0.0
    TP_cat = 0
    FP_cat = 0
    FN_cat = 0

    D_max = math.sqrt(128)

    # Evaluate each sample
    for sample_id in gold_data:
        if sample_id not in pred_data:
            gold_triplets = gold_data[sample_id].get('Quadruplet', [])
            FN_cat += len(gold_triplets)
            continue

        gold_triplets = gold_data[sample_id].get('Quadruplet', [])
        pred_triplets = pred_data[sample_id].get('Triplet', [])

        matched_pred = set()

        for gold_trip in gold_triplets:
            gold_aspect = gold_trip['Aspect'].lower().strip()
            gold_opinion = gold_trip['Opinion'].lower().strip()

            try:
                gold_v, gold_a = map(float, gold_trip['VA'].split('#'))
            except:
                continue

            best_ctp = 0.0
            best_idx = -1

            for i, pred_trip in enumerate(pred_triplets):
                if i in matched_pred:
                    continue

                pred_aspect = pred_trip['Aspect'].lower().strip()
                pred_opinion = pred_trip['Opinion'].lower().strip()

                if pred_aspect != gold_aspect or pred_opinion != gold_opinion:
                    continue

                try:
                    pred_v, pred_a = map(float, pred_trip['VA'].split('#'))
                except:
                    continue

                if not (1.0 <= pred_v <= 9.0 and 1.0 <= pred_a <= 9.0):
                    continue

                distance = math.sqrt((pred_v - gold_v)**2 + (pred_a - gold_a)**2)
                ctp = max(0.0, 1.0 - (distance / D_max))

                if ctp > best_ctp:
                    best_ctp = ctp
                    best_idx = i

            if best_idx >= 0:
                matched_pred.add(best_idx)
                TP_cat += 1
                cTP_total += best_ctp
            else:
                FN_cat += 1

        FP_cat += len(pred_triplets) - len(matched_pred)

    # Calculate metrics
    cPrecision = cTP_total / (TP_cat + FP_cat) if (TP_cat + FP_cat) > 0 else 0.0
    cRecall = cTP_total / (TP_cat + FN_cat) if (TP_cat + FN_cat) > 0 else 0.0
    cF1 = 2 * cPrecision * cRecall / (cPrecision + cRecall) if (cPrecision + cRecall) > 0 else 0.0

    print("EVALUATION RESULTS")
    print(f"True Positives (TP):      {TP_cat}")
    print(f"Continuous TP (cTP):      {cTP_total:.4f}")
    print(f"False Positives (FP):     {FP_cat}")
    print(f"False Negatives (FN):     {FN_cat}")
    print(f"{'─'*80}")
    print(f"cPrecision:               {cPrecision:.4f}")
    print(f"cRecall:                  {cRecall:.4f}")
    print(f"cF1:                      {cF1:.4f}")


    return {
        'cPrecision': cPrecision,
        'cRecall': cRecall,
        'cF1': cF1,
        'TP': TP_cat,
        'cTP': cTP_total,
        'FP': FP_cat,
        'FN': FN_cat
    }


def create_domain_download_package(domain: str, config, eval_metrics: dict):
    """
    Create complete download package for a domain immediately after evaluation
    """

    print(f"CREATING DOWNLOAD PACKAGE - {domain.upper()}")
    # Create domain-specific download directory
    download_dir = f"/kaggle/working/download_{domain}"
    os.makedirs(download_dir, exist_ok=True)

    domain_output_dir = config.get_output_dir(domain)
    files_packaged = []

    # 1. TEST PREDICTIONS (untuk submission)
    test_pred = os.path.join(domain_output_dir, f"pred_eng_{domain}.jsonl")
    if os.path.exists(test_pred):
        shutil.copy(test_pred, os.path.join(download_dir, f"{domain}_test_predictions.jsonl"))
        files_packaged.append(f"{domain}_test_predictions.jsonl")

    # 2. EVALUATION RESULTS (Train Data)
    # Predictions
    eval_pred = os.path.join(domain_output_dir, "eval_pred_full.jsonl")
    if os.path.exists(eval_pred):
        shutil.copy(eval_pred, os.path.join(download_dir, f"{domain}_eval_pred_full.jsonl"))
        files_packaged.append(f"{domain}_eval_pred_full.jsonl")

    # Gold
    eval_gold = os.path.join(domain_output_dir, "eval_gold_full.jsonl")
    if os.path.exists(eval_gold):
        shutil.copy(eval_gold, os.path.join(download_dir, f"{domain}_eval_gold_full.jsonl"))
        files_packaged.append(f"{domain}_eval_gold_full.jsonl")

    # Metrics JSON
    metrics_file = os.path.join(download_dir, f"{domain}_metrics.json")
    with open(metrics_file, 'w') as f:
        json.dump({
            'domain': domain,
            'metrics': eval_metrics,
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
        }, f, indent=2)
    files_packaged.append(f"{domain}_metrics.json")

    # 3. TOKEN ANALYSIS (if exists)
    token_analysis = "/kaggle/working/token_analysis_results.json"
    if os.path.exists(token_analysis):
        shutil.copy(token_analysis, os.path.join(download_dir, "token_analysis_results.json"))
        files_packaged.append("token_analysis_results.json")

    # 4. CREATE SUBMISSION ZIP for this domain
    submission_zip = os.path.join(download_dir, f"{config.SUBTASK}_{domain}_submission.zip")
    with zipfile.ZipFile(submission_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        if os.path.exists(test_pred):
            arcname = os.path.join(config.SUBTASK, os.path.basename(test_pred))
            zipf.write(test_pred, arcname)
    files_packaged.append(f"{config.SUBTASK}_{domain}_submission.zip")

    # 5. CREATE MASTER ZIP (all files for this domain)
    master_zip = f"/kaggle/working/{domain}_all_files.zip"
    shutil.make_archive(
        master_zip.replace('.zip', ''),
        'zip',
        download_dir
    )

    # DISPLAY DOWNLOAD LINKS
    print(f"FILES READY FOR DOWNLOAD:")
    for fname in files_packaged:
        fpath = os.path.join(download_dir, fname)
        if os.path.exists(fpath):
            size_kb = os.path.getsize(fpath) / 1024
            print(f"  • {fname} ({size_kb:.1f} KB)")

    print(f"\nDOWNLOAD MASTER ZIP (contains all {len(files_packaged)} files):")
    print(f"   Size: {os.path.getsize(master_zip)/1024:.1f} KB")

    # Display download link
    display(FileLink(master_zip))
    return master_zip

def run_full_evaluation_with_auto_save(domain: str, train_data: List[Dict],
                                       model, tokenizer, config):
    """
    Run evaluation and immediately create download package
    """
    print(f"FULL EVALUATION - {domain.upper()}")
    print(f"Evaluating on {len(train_data)} samples")

    # Prepare output files
    domain_output_dir = config.get_output_dir(domain)
    eval_gold_file = os.path.join(domain_output_dir, "eval_gold_full.jsonl")
    eval_pred_file = os.path.join(domain_output_dir, "eval_pred_full.jsonl")

    # Save gold file
    print("\nSaving gold annotations")
    with open(eval_gold_file, 'w', encoding='utf-8') as f:
        for sample in train_data:
            f.write(json.dumps(sample, ensure_ascii=False) + '\n')

    # Set model to inference mode
    FastLanguageModel.for_inference(model)
    model.eval()

    # Run inference
    print(f"\nRunning inference on {len(train_data)} samples")

    eval_results = []
    failed_samples = 0

    from tqdm.auto import tqdm
    start_time = time.time()

    for sample in tqdm(train_data, desc=f"{domain.capitalize()} eval"):
        try:
            text = sample['Text']
            prompt = create_prompt(text, config.SELECTED_TEMPLATE)

            messages = [{"role": "user", "content": prompt}]
            formatted = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = tokenizer(
                formatted,
                return_tensors="pt",
                truncation=True,
                max_length=config.MAX_SEQ_LENGTH,
                padding=True
            ).to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    input_ids=inputs['input_ids'],
                    attention_mask=inputs['attention_mask'],
                    max_new_tokens=256,
                    temperature=0.7,
                    top_p=0.9,
                    do_sample=True,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )

            response = tokenizer.decode(outputs[0], skip_special_tokens=True)
            answer = response.split("assistant")[-1].strip() if "assistant" in response else response
            triplets = extract_triplets(answer)

            eval_results.append({
                "ID": sample['ID'],
                "Triplet": triplets
            })

        except Exception as e:
            failed_samples += 1
            eval_results.append({
                "ID": sample['ID'],
                "Triplet": []
            })

    elapsed_time = time.time() - start_time
    minutes = int(elapsed_time // 60)
    seconds = int(elapsed_time % 60)

    print(f"\n  Inference completed in {minutes}m {seconds}s")
    if failed_samples > 0:
        print(f"⚠️  Failed samples: {failed_samples}")

    # Save predictions
    with open(eval_pred_file, 'w', encoding='utf-8') as f:
        for r in eval_results:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

    # Run evaluation
    eval_metrics = evaluate_task2(eval_gold_file, eval_pred_file)

    # Save metrics to file
    metrics_file = os.path.join(domain_output_dir, "eval_metrics_full.json")
    with open(metrics_file, 'w', encoding='utf-8') as f:
        json.dump({
            'domain': domain,
            'samples': len(train_data),
            'failed_samples': failed_samples,
            'inference_time_seconds': elapsed_time,
            'metrics': eval_metrics
        }, f, indent=2)


    print(f"  {domain.upper()} FINAL METRICS")

    print(f"Dataset size: {len(train_data)}")
    print(f"cPrecision:   {eval_metrics['cPrecision']:.4f}")
    print(f"cRecall:      {eval_metrics['cRecall']:.4f}")
    print(f"cF1:          {eval_metrics['cF1']:.4f} ⭐")

    # IMMEDIATELY CREATE DOWNLOAD PACKAGE
    master_zip = create_domain_download_package(domain, config, eval_metrics)
    return eval_metrics, master_zip

# RUN RESTAURANT - AUTO-SAVE & DOWNLOAD
print("\nRESTAURANT - FULL EVALUATION WITH AUTO-DOWNLOAD")
restaurant_eval_metrics, restaurant_zip = run_full_evaluation_with_auto_save(
    domain='restaurant',
    train_data=clean_restaurant_data,
    model=restaurant_model,
    tokenizer=tokenizer,
    config=config
)

# RUN LAPTOP - AUTO-SAVE & DOWNLOAD
print("\nLAPTOP - FULL EVALUATION WITH AUTO-DOWNLOAD")
laptop_eval_metrics, laptop_zip = run_full_evaluation_with_auto_save(
    domain='laptop',
    train_data=clean_laptop_data,
    model=laptop_model,
    tokenizer=tokenizer,
    config=config
)

# FINAL SUMMARY
print("FINAL EXECUTION SUMMARY - DUAL DOMAIN")

print(f"\n  TASK INFORMATION:")
print(f"   Task: {config.TASK}")
print(f"   Subtask: {config.SUBTASK}")
print(f"   Language: {config.LANGUAGE}")
print(f"   Domains: Restaurant, Laptop")

print("  DATASET STATISTICS")
print(f"\n{'Domain':<15} {'Train (Initial)':<20} {'Train (Cleaned)':<20} {'Test':<15}")
print(f"{'Restaurant':<15} {len(restaurant_train_data):<20} {len(clean_restaurant_data):<20} {len(restaurant_test_data):<15}")
print(f"{'Laptop':<15} {len(laptop_train_data):<20} {len(clean_laptop_data):<20} {len(laptop_test_data):<15}")

print("  MODEL & TRAINING CONFIGURATION")
print(f"   Model: {config.MODEL_NAME}")
print(f"   Prompt template: {config.SELECTED_TEMPLATE}")
print(f"   Max sequence length: {config.MAX_SEQ_LENGTH}")
print(f"   LoRA: R={config.LORA_R}, Alpha={config.LORA_ALPHA}, Dropout={config.LORA_DROPOUT}")
print(f"   Batch size: {config.BATCH_SIZE}")
print(f"   Gradient accumulation: {config.GRADIENT_ACCUMULATION_STEPS}")
print(f"   Effective batch size: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS}")
print(f"   Epochs: {config.EPOCHS}")
print(f"   Learning rate: {config.LEARNING_RATE}")

print("   TRAINING TIME")
print(f"\n{'Domain':<15} {'Duration':<20} {'Steps':<10} {'Final Loss':<15}")
print(f"{'Restaurant':<15} {restaurant_stats['duration_formatted']:<20} {restaurant_stats['steps']:<10} {restaurant_stats['final_loss']:<15.4f}")
print(f"{'Laptop':<15} {laptop_stats['duration_formatted']:<20} {laptop_stats['steps']:<10} {laptop_stats['final_loss']:<15.4f}")

total_seconds = restaurant_stats['duration_seconds'] + laptop_stats['duration_seconds']
total_hours = int(total_seconds // 3600)
total_minutes = int((total_seconds % 3600) // 60)
total_seconds_rem = int(total_seconds % 60)
print(f"\n   Total training time: {total_hours}h {total_minutes}m {total_seconds_rem}s")

print("  EVALUATION RESULTS (Full Training Dataset)")
print(f"\n{'Domain':<15} {'cPrecision':<15} {'cRecall':<15} {'cF1':<15}")
print(f"{'Restaurant':<15} {restaurant_eval_metrics['cPrecision']:<15.4f} {restaurant_eval_metrics['cRecall']:<15.4f} {restaurant_eval_metrics['cF1']:<15.4f}")
print(f"{'Laptop':<15} {laptop_eval_metrics['cPrecision']:<15.4f} {laptop_eval_metrics['cRecall']:<15.4f} {laptop_eval_metrics['cF1']:<15.4f}")

avg_cf1 = (restaurant_eval_metrics['cF1'] + laptop_eval_metrics['cF1']) / 2
print(f"\n     Average cF1: {avg_cf1:.4f}")

print("  DOWNLOAD YOUR RESULTS")
print("\nRestaurant ZIP:")
display(FileLink(restaurant_zip))
print("\nLaptop ZIP:")
display(FileLink(laptop_zip))



RESTAURANT - FULL EVALUATION WITH AUTO-DOWNLOAD



FULL EVALUATION - RESTAURANT
Evaluating on 2276 samples

Saving gold annotations...

Running inference on 2276 samples...


Restaurant eval:   0%|          | 0/2276 [00:00<?, ?it/s]


✓ Inference completed in 63m 27s

────────────────────────────────────────────────────────────────────────────────
Running evaluation metrics...
────────────────────────────────────────────────────────────────────────────────
Gold samples: 2276
Pred samples: 2276

EVALUATION RESULTS
True Positives (TP):      2426
Continuous TP (cTP):      2248.6076
False Positives (FP):     1286
False Negatives (FN):     1225
────────────────────────────────────────────────────────────────────────────────
cPrecision:               0.6058
cRecall:                  0.6159
cF1:                      0.6108

✅ RESTAURANT FINAL METRICS
Dataset size: 2276
cPrecision:   0.6058
cRecall:      0.6159
cF1:          0.6108 ⭐

CREATING DOWNLOAD PACKAGE - RESTAURANT
✓ Test predictions
✓ Evaluation predictions (train)
✓ Evaluation gold (train)
✓ Metrics JSON
✓ Token analysis
✓ Submission ZIP
✓ Master ZIP created

────────────────────────────────────────────────────────────────────────────────
FILES READY FOR DOWNLOAD

/kaggle/working/restaurant_all_files.zip


✅ RESTAURANT COMPLETE - FILES READY!
Download link displayed above


LAPTOP - FULL EVALUATION WITH AUTO-DOWNLOAD



FULL EVALUATION - LAPTOP
Evaluating on 4052 samples

Saving gold annotations...

Running inference on 4052 samples...


...

/kaggle/working/laptop_all_files.zip


✅ LAPTOP COMPLETE - FILES READY!
Download link displayed above


FINAL EXECUTION SUMMARY - DUAL DOMAIN

📋 TASK INFORMATION:
   Task: task2
   Subtask: subtask_2
   Language: eng
   Domains: Restaurant, Laptop

📊 DATASET STATISTICS

Domain          Train (Initial)      Train (Cleaned)      Test           
────────────────────────────────────────────────────────────────────────────────
Restaurant      2284                 2276                 200            
Laptop          4076                 4052                 200            

🤖 MODEL & TRAINING CONFIGURATION
   Model: unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit
   Prompt template: one_shot_instruction
   Max sequence length: 232
   LoRA: R=16, Alpha=32, Dropout=0.0
   Batch size: 1
   Gradient accumulation: 16
   Effective batch size: 16
   Epochs: 3
   Learning rate: 0.0001

⏱️  TRAINING TIME

Domain          Duration             Steps      Final Loss     
───────────────────────────────────────────────────────────────────────────────

/kaggle/working/restaurant_all_files.zip


Laptop ZIP:


/kaggle/working/laptop_all_files.zip